Navpreet Kloy, Logan Cheng


Dataset URL: https://www.kaggle.com/datasets/zkskhurram/appendicitis-comprehensive-clinical-dataset/data



In [43]:
# import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

In [19]:
# load dataset
df = pd.read_csv("appendicitis_comprehensive_dataset.csv")
df.head()

,Patient_ID,Age,Gender,BMI,Is_Pregnant,Duration_of_Symptoms_Hours,Pain_Migration,Abdominal_Pain_Location,Nausea_Vomiting,Loss_of_Appetite,...,Rovsing_Sign,Psoas_Sign,WBC_Count_k_uL,Neutrophil_Percentage,CRP_Level_mg_L,Ultrasound_Findings,Pathological_Cause,Severity,Management,Final_Diagnosis
0,APX-0001,32,Female,23.4,No,17,Yes,RLQ,Yes,No,...,Yes,Yes,11.5,77.8,63.2,Non-visualized,Fecalith/Appendicolith,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
1,APX-0002,5,Male,19.4,No,31,No,RLQ,No,No,...,Yes,No,12.6,74.7,89.1,Target Sign,Lymphoid Hyperplasia,Uncomplicated,Antibiotics (Conservative),Appendicitis
2,APX-0003,28,Female,27.5,No,49,Yes,RLQ,Yes,Yes,...,No,Yes,16.2,77.0,92.9,Periappendiceal Fluid,Lymphoid Hyperplasia,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
3,APX-0004,20,Male,28.7,No,32,No,RLQ,Yes,Yes,...,Yes,Yes,19.1,88.3,12.7,Periappendiceal Fluid,Fecalith/Appendicolith,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
4,APX-0005,12,Female,22.0,No,47,No,RLQ,No,No,...,Yes,No,12.2,65.0,8.8,Normal,NaN,NaN,Observation/Other,Other (Ectopic Pregnancy (if pregnant))


In [20]:
# dataset structure overiew including data types and missing values
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Patient_ID                  1500 non-null   object 
 1   Age                         1500 non-null   int64  
 2   Gender                      1500 non-null   object 
 3   BMI                         1500 non-null   float64
 4   Is_Pregnant                 1500 non-null   object 
 5   Duration_of_Symptoms_Hours  1500 non-null   int64  
 6   Pain_Migration              1500 non-null   object 
 7   Abdominal_Pain_Location     1500 non-null   object 
 8   Nausea_Vomiting             1500 non-null   object 
 9   Loss_of_Appetite            1500 non-null   object 
 10  Fever_Temp_C                1500 non-null   float64
 11  Rebound_Tenderness          1500 non-null   object 
 12  McBurney_Sign               1500 non-null   object 
 13  Rovsing_Sign                1500 

In [21]:
# check categorical values
for col in df.select_dtypes(include='object').columns:
    print(df[col].value_counts())
    print()

Patient_ID
APX-1500    1
APX-0001    1
APX-0002    1
APX-0003    1
APX-0004    1
           ..
APX-0017    1
APX-0016    1
APX-0015    1
APX-0014    1
APX-0013    1
Name: count, Length: 1500, dtype: int64

Gender
Male      815
Female    685
Name: count, dtype: int64

Is_Pregnant
No     1485
Yes      15
Name: count, dtype: int64

Pain_Migration
Yes    880
No     620
Name: count, dtype: int64

Abdominal_Pain_Location
RLQ              1058
Periumbilical     192
Generalized       116
LLQ                51
Epigastric         42
Pelvic             35
RUQ                 6
Name: count, dtype: int64

Nausea_Vomiting
Yes    1081
No      419
Name: count, dtype: int64

Loss_of_Appetite
Yes    1067
No      433
Name: count, dtype: int64

Rebound_Tenderness
Yes    1021
No      479
Name: count, dtype: int64

McBurney_Sign
Yes    1085
No      415
Name: count, dtype: int64

Rovsing_Sign
No     766
Yes    734
Name: count, dtype: int64

Psoas_Sign
No     983
Yes    517
Name: count, dtype: int64

Ultrasou

In [22]:
# display all column names in the dataset
print(df.columns.tolist())

['Patient_ID', 'Age', 'Gender', 'BMI', 'Is_Pregnant', 'Duration_of_Symptoms_Hours', 'Pain_Migration', 'Abdominal_Pain_Location', 'Nausea_Vomiting', 'Loss_of_Appetite', 'Fever_Temp_C', 'Rebound_Tenderness', 'McBurney_Sign', 'Rovsing_Sign', 'Psoas_Sign', 'WBC_Count_k_uL', 'Neutrophil_Percentage', 'CRP_Level_mg_L', 'Ultrasound_Findings', 'Pathological_Cause', 'Severity', 'Management', 'Final_Diagnosis']


In [23]:
# view the final diagnosis values and their class distributions
df["Final_Diagnosis"].value_counts()

,count
Final_Diagnosis,
Appendicitis,1190
Other (Gastroenteritis),76
Other (Mesenteric Adenitis),67
Other (UTI),61
Other (Kidney Stones),38
Other (PID),26
Other (Ovarian Cyst Rupture),24
Other (Ectopic Pregnancy (if pregnant)),18


In [24]:
# Convert string values to numeric using LabelEncoder

# create an instance of LableEncoder
le = LabelEncoder()

# list of categorical columns to encode
cols = ["Gender", "Is_Pregnant","Pain_Migration", "Abdominal_Pain_Location","Nausea_Vomiting", "Loss_of_Appetite",
        "Rebound_Tenderness", "McBurney_Sign", "Rovsing_Sign", "Psoas_Sign"]

for col in cols:
    df[col] = le.fit_transform(df[col])
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))  # create a dictionary showing mapping
    print(col, mapping)

df # view encoded categorical features

Gender {'Female': np.int64(0), 'Male': np.int64(1)}
Is_Pregnant {'No': np.int64(0), 'Yes': np.int64(1)}
Pain_Migration {'No': np.int64(0), 'Yes': np.int64(1)}
Abdominal_Pain_Location {'Epigastric': np.int64(0), 'Generalized': np.int64(1), 'LLQ': np.int64(2), 'Pelvic': np.int64(3), 'Periumbilical': np.int64(4), 'RLQ': np.int64(5), 'RUQ': np.int64(6)}
Nausea_Vomiting {'No': np.int64(0), 'Yes': np.int64(1)}
Loss_of_Appetite {'No': np.int64(0), 'Yes': np.int64(1)}
Rebound_Tenderness {'No': np.int64(0), 'Yes': np.int64(1)}
McBurney_Sign {'No': np.int64(0), 'Yes': np.int64(1)}
Rovsing_Sign {'No': np.int64(0), 'Yes': np.int64(1)}
Psoas_Sign {'No': np.int64(0), 'Yes': np.int64(1)}


,Patient_ID,Age,Gender,BMI,Is_Pregnant,Duration_of_Symptoms_Hours,Pain_Migration,Abdominal_Pain_Location,Nausea_Vomiting,Loss_of_Appetite,...,Rovsing_Sign,Psoas_Sign,WBC_Count_k_uL,Neutrophil_Percentage,CRP_Level_mg_L,Ultrasound_Findings,Pathological_Cause,Severity,Management,Final_Diagnosis
0,APX-0001,32,0,23.4,0,17,1,5,1,0,...,1,1,11.5,77.8,63.2,Non-visualized,Fecalith/Appendicolith,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
1,APX-0002,5,1,19.4,0,31,0,5,0,0,...,1,0,12.6,74.7,89.1,Target Sign,Lymphoid Hyperplasia,Uncomplicated,Antibiotics (Conservative),Appendicitis
2,APX-0003,28,0,27.5,0,49,1,5,1,1,...,0,1,16.2,77.0,92.9,Periappendiceal Fluid,Lymphoid Hyperplasia,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
3,APX-0004,20,1,28.7,0,32,0,5,1,1,...,1,1,19.1,88.3,12.7,Periappendiceal Fluid,Fecalith/Appendicolith,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
4,APX-0005,12,0,22.0,0,47,0,5,0,0,...,1,0,12.2,65.0,8.8,Normal,NaN,NaN,Observation/Other,Other (Ectopic Pregnancy (if pregnant))
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,APX-1496,36,0,19.4,0,22,0,1,1,1,...,1,1,16.6,77.3,60.5,Non-visualized,Tumor,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
1496,APX-1497,27,1,27.7,0,13,1,5,0,0,...,0,0,14.6,75.9,41.7,Periappendiceal Fluid,Fecalith/Appendicolith,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
1497,APX-1498,20,1,33.1,0,43,1,5,1,0,...,1,0,20.3,67.8,40.1,Enlarged (>6mm),Lymphoid Hyperplasia,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
1498,APX-1499,16,1,24.6,0,80,0,5,1,1,...,1,1,15.2,84.3,82.6,Appendicolith Seen,Lymphoid Hyperplasia,Uncomplicated,Laparoscopic Appendectomy,Appendicitis


In [25]:
# convert the target variable to a binary value where 1 = Appendicitis and 0 = the other diagnoses
df["Final_Diagnosis"] = (df["Final_Diagnosis"] == "Appendicitis").astype(int)

In [26]:
# check the new class distribution for the target variable
df["Final_Diagnosis"].value_counts()

,count
Final_Diagnosis,
1,1190
0,310


In [27]:
# split dataset in features and target variable
x_data=df[['Age','Gender','BMI','Is_Pregnant','Duration_of_Symptoms_Hours','Pain_Migration','Abdominal_Pain_Location','Nausea_Vomiting','Loss_of_Appetite','Fever_Temp_C','Rebound_Tenderness','McBurney_Sign','Rovsing_Sign','Psoas_Sign','WBC_Count_k_uL','Neutrophil_Percentage','CRP_Level_mg_L']]
y_data=df['Final_Diagnosis']

In [28]:
# Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.3,stratify=y_data,random_state=1)

In [29]:
# Train a single Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train,y_train)

# Calculate the accuracy and assign it to dt_accuracy
dt_accuracy = dt_model.score(X_test, y_test)

print("Decision Tree Test Accuracy:", f"{dt_accuracy * 100:.2f}%")


Decision Tree Test Accuracy: 97.11%


In [30]:
# use GridSearchCV to find best Random forest hyperparameters

param_grid = {
    'n_estimators': [50,100,200],
    'max_features': ['sqrt','log2',None,2],
    'max_depth': [None,10,20]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5)
grid.fit(X_train, y_train)

print(grid.best_params_)

{'max_depth': None, 'max_features': 2, 'n_estimators': 100}


In [31]:
# Train a Random Forest with random feature selection
rf_model = RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42)  # max_features='sqrt'  means roughly sqrt(total_features) are considered at each split (about 4 in this dataset)

rf_model.fit(X_train, y_train)

# Calculate the accuracy and assign it to rf_accuracy
rf_accuracy = rf_model.score(X_test, y_test)

# Now you can use rf_accuracy in the print statements below
print("RF Model Accuracy:", f"{rf_accuracy * 100:.2f}", "%")


RF Model Accuracy: 99.56 %


In [32]:
# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)

# Calculate precision, recall, and F1-score for Random Forest
precision_rf = metrics.precision_score(y_test, y_pred_rf)
recall_rf = metrics.recall_score(y_test, y_pred_rf)
f1_rf = metrics.f1_score(y_test, y_pred_rf)


print("RF Precision:", precision_rf)
print("RF Recall:", recall_rf)
print("RF F1-Score:", f1_rf)


RF Precision: 0.9971988795518207
RF Recall: 0.9971988795518207
RF F1-Score: 0.9971988795518207


In [33]:
# For the feature importance, show a table of features and its corresponding values

feature_importances = pd.DataFrame({'feature': x_data.columns, 'importance': rf_model.feature_importances_})
feature_importances = feature_importances.sort_values('importance', ascending=False)
feature_importances

,feature,importance
15,Neutrophil_Percentage,0.332000
14,WBC_Count_k_uL,0.190346
16,CRP_Level_mg_L,0.107886
6,Abdominal_Pain_Location,0.092811
10,Rebound_Tenderness,0.067019
11,McBurney_Sign,0.052938
9,Fever_Temp_C,0.033336
8,Loss_of_Appetite,0.031195
5,Pain_Migration,0.030839
12,Rovsing_Sign,0.016355


In [42]:
# Prepare a provision for a new data entry and its prediction of an outcome.

def predict_diabetes():
    while True:
        try:
            age = int(input("Enter age: "))
            gender = int(input("Enter 0 for female or 1 for male: "))
            BMI = float(input("Enter BMI: "))
            Is_Pregnant = int(input("Enter pregnancy status (0: Not Pregnant or 1:Pregnant): "))
            Duration_of_Symptoms_Hours = int(input("Enter duration of symptoms in hours: "))
            Pain_Migration = int(input("Enter whether pain has migrated (0: No or 1: Yes) : "))
            Abdominal_Pain_Location = int(input("Enter abdominal pain location (Epigastric: 0, Generalized: 1, LLQ: 2, Pelvic: 3, Periumbilical: 4, RLQ: 5, RUQ: 6): "))
            Nausea_Vomiting = int(input("Enter whether there is nausea vomiting (0: No or 1: Yes): "))
            Loss_of_Appetite = int(input("Enter whether there is loss appetite (0: No or 1: Yes): "))
            Fever_Temp_C = float(input("Enter body temperature in °C: "))
            Rebound_Tenderness = int(input("Enter whether there is rebound tenderness (0: No or 1:Yes): "))
            McBurney_Sign = int(input("Enter whether there is a McBurney sign (0:No or 1:Yes): "))
            Rovsing_Sign = int(input("Enter whether there is a Rovsing sign (0:No or 1:Yes): "))
            Psoas_Sign = int(input("Enter whether there is a Psoas sign (0:No or 1:Yes): "))
            WBC_Count_k_uL = float(input("Enter white blood cell count (k/µL): "))
            Neutrophil_Percentage = float(input("Enter neutrophil percentage (%): "))
            CRP_Level_mg_L = float(input("Enter CRP level (mg/L): "))


            # input validation for non-negative values
            if any(val < 0 for val in [age, BMI, Duration_of_Symptoms_Hours, Fever_Temp_C, WBC_Count_k_uL, Neutrophil_Percentage, CRP_Level_mg_L ]):
              print("Error: Please enter non-negative value.")
              continue  #restart the loop

            # input validation for binary variables (0 = No or 1 = Yes)
            if any(val not in [0,1] for val in [gender, Is_Pregnant, Pain_Migration, Nausea_Vomiting, Loss_of_Appetite, Rebound_Tenderness, McBurney_Sign, Rovsing_Sign, Psoas_Sign]):
              print("Error: Please enter 0 or 1.")
              continue  #restart the loop

            # input validation for encoded abdominal pain location values (0–5)
            if Abdominal_Pain_Location not in [0,1,2,3,4,5]:
              print("Error: Please enter corresponding values between 0 adn 5 for abominal pain location.")
              continue  #restart the loop


            new_patient_data = pd.DataFrame([[age, gender, BMI, Is_Pregnant, Duration_of_Symptoms_Hours, Pain_Migration, Abdominal_Pain_Location, Nausea_Vomiting, Loss_of_Appetite, Fever_Temp_C, Rebound_Tenderness, McBurney_Sign, Rovsing_Sign, Psoas_Sign, WBC_Count_k_uL, Neutrophil_Percentage, CRP_Level_mg_L]],
                                           columns=['Age','Gender','BMI','Is_Pregnant','Duration_of_Symptoms_Hours','Pain_Migration','Abdominal_Pain_Location','Nausea_Vomiting','Loss_of_Appetite','Fever_Temp_C','Rebound_Tenderness','McBurney_Sign','Rovsing_Sign','Psoas_Sign','WBC_Count_k_uL','Neutrophil_Percentage','CRP_Level_mg_L'])

            prediction =rf_model.predict(new_patient_data)[0]

            if prediction == 1:
                print("Prediction: Patient is likely to have appendicitis.")
            else:
                print("Prediction: Patient is not likely not to have appendicitis.")

            another_patient = input("Enter data for another patient? (yes/no): ")
            if another_patient.lower() != 'yes':
                break #exit loop
        except ValueError:
            print("Invalid input. Please enter valid numerical values.")

predict_diabetes()

Enter age: 20
Enter 0 for female or 1 for male: 1
Enter BMI: 25
Enter pregnancy status (0: Not Pregnant or 1:Pregnant): 0
Enter duration of symptoms in hours: 5
Enter whether pain has migrated (0: No or 1: Yes) : 1
Enter abdominal pain location (Epigastric: 0, Generalized: 1, LLQ: 2, Pelvic: 3, Periumbilical: 4, RLQ: 5, RUQ: 6): 4
Enter whether there is nausea vomiting (0: No or 1: Yes): 0
Enter whether there is loss appetite (0: No or 1: Yes): 0
Enter body temperature in °C: 36
Enter whether there is rebound tenderness (0: No or 1:Yes): 1
Enter whether there is a McBurney sign (0:No or 1:Yes): 1
Enter whether there is a Rovsing sign (0:No or 1:Yes): 1
Enter whether there is a Psoas sign (0:No or 1:Yes): 0
Enter white blood cell count (k/µL): 5
Enter neutrophil percentage (%): 50
Enter CRP level (mg/L): 5
Prediction: Patient is not likely not to have appendicitis.
Enter data for another patient? (yes/no): no


The Appendicitis Clinical Dataset was used to build a model to determine whether the patient had appendicitis or other diagnosis. The dataset included patient demographics and related health symptoms, laboratory data, and findings for each patient. The dataset had a total of 1,500 patient data. A random forest algorithm was used to develop the prediction model. A random forest is similar to a decision tree, except instead of one decision tree, it builds multiple decision trees. The final prediction is an average of all tree predictions. The dataset originally had multiple final diagnoses, these were feature engineered so that other diagnosis (0)  represents all other possible outcomes. The prediction was made a binary value with prediction either being appendicitis (1) or other diagnosis (0). Then a total of 17 predictor variables were carefully selected out of 23 variables the dataset contained. The dataset was then split into a test and training set. First, a decision tree model was created, which has an accuracy of 97.11%, meaning the predictions matched the actual outcome in the test 97.11% of the time. Next, a random forest model was developed, which has an accuracy of 99.56%. Both models have high accuracy, with random forest being slightly better. The random forest model was further analyzed using precision, recall, and F1-score metrics. Scores across the three metrics are the same, each metric value has a value of 99.72%. The random forest model correctly predicts appendicitis 99.72% of the time (precision) and identifies 99.72% of appendicitis cases in the model (recall). The final F1-score of 99.72%, again demonstrates the random forest model has a strong performance and indicates it is a good model to predict whether a patient has appendicitis based on given the variables. The top 3 important features were Neutrophil_Percentage, WBC_Count, and CRP_Level, these had the most impact on the outcome of the model’s prediction.
